# FashionSense — Exploratory Data Analysis
Analyse the sampled DeepFashion2 dataset before training.

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image

SAMPLE_DIR = Path('../data/sample_dataset')
ANNO_DIR   = SAMPLE_DIR / 'annos'
IMG_DIR    = SAMPLE_DIR / 'images'

CATEGORY_MAP = {
    1:'short_sleeve_top', 2:'long_sleeve_top', 3:'short_sleeve_outwear',
    4:'long_sleeve_outwear', 5:'vest', 6:'sling', 7:'shorts',
    8:'trousers', 9:'skirt', 10:'short_sleeve_dress',
    11:'long_sleeve_dress', 12:'vest_dress', 13:'sling_dress'
}

## 1. Class Distribution

In [ ]:
cat_counts = Counter()
rows = []

for fpath in ANNO_DIR.glob('*.json'):
    with open(fpath) as f:
        anno = json.load(f)
    for key, val in anno.items():
        if key.startswith('item') and isinstance(val, dict):
            cat_id = val.get('category_id')
            if cat_id:
                cat_counts[cat_id] += 1
                rows.append({'cat_id': cat_id, 'cat_name': CATEGORY_MAP.get(cat_id,'?'), 'img': fpath.stem})

df = pd.DataFrame(rows)
print(df['cat_name'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
vc = df['cat_name'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(vc)))
ax.bar(vc.index, vc.values, color=colors)
ax.set_title('Category Distribution in Sample Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../docs/class_distribution.png', dpi=150)
plt.show()

## 2. Bounding Box Size Analysis

In [ ]:
bbox_areas = []
for fpath in list(ANNO_DIR.glob('*.json'))[:2000]:
    with open(fpath) as f:
        anno = json.load(f)
    img_path = IMG_DIR / f'{fpath.stem}.jpg'
    if not img_path.exists():
        continue
    with Image.open(img_path) as im:
        W, H = im.size
    for key, val in anno.items():
        if key.startswith('item') and isinstance(val, dict):
            bb = val.get('bounding_box')
            if bb and len(bb) == 4:
                x1,y1,x2,y2 = bb
                rel_area = ((x2-x1) * (y2-y1)) / (W * H)
                bbox_areas.append(rel_area)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(bbox_areas, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Relative Bounding Box Area Distribution')
ax.set_xlabel('Relative area (bbox / image)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(f'Median relative area: {np.median(bbox_areas):.3f}')

## 3. Sample Image Grid

In [ ]:
import random
sample_imgs = random.sample(list(IMG_DIR.glob('*.jpg')), 16)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(img_path.stem, fontsize=7)

plt.suptitle('Sample Images from Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()